# 05 · ALS và vector phiên (ticket #14)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/als_{val,test}_run0.parquet`
- `item_embeddings/als.npy`, `item_embeddings/als_tracks.parquet`
- `als_params.json`: tham số đã chọn và NDCG@10 trên val của cả lưới (lần chạy 1, 2 ở #26 dùng lại file này qua `--params`)

Lưới chọn trên val: factors {64, 128} × regularization {0,01; 0,1} × decay {0,5; 0,7; 0,9; 1,0} (16 cấu hình, 4 lần học ALS). Chọn theo NDCG@10.

Bật **Internet** (clone repo, `pip install implicit`). Chỉ cần CPU. Ước tính 20–30 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #14 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
# implicit không có sẵn trên Kaggle (#2)
!pip install -q implicit==0.7.3

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.als import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # chọn tham số trên val, xuất file, có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lưới trên val, xếp theo NDCG@10, so với hai baseline của #9 (notebook 04, val 9.212 phiên)
import pandas as pd

BASELINE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297}
grid = pd.DataFrame(report["grid"]).sort_values("ndcg", ascending=False)
print(grid.round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nChọn: {report['params']} | NDCG@10 val {ndcg:.4f}")
for name, base in BASELINE_NDCG.items():
    print(f"  so với {name} {base}: {ndcg / base - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Theo `docs/data-contract.md`, output chia về hai Dataset:
   - Tab Output của version vừa chạy → **New Dataset** tên `mixnext-embeddings` (gồm `item_embeddings/als.*` và `als_params.json`). Embedding này cho app (#20) và sinh chuỗi (#18); item2vec (#15) thêm vào version sau.
   - Tải hai file `candidates/als_{val,test}_run0.parquet` ở tab Output, rồi mở Dataset `mixnext-candidates` → **New Version** → upload vào thư mục `candidates/`, giữ các file của Popularity và item-kNN.

## Kết quả trên val (chưa chạy)

Dán bảng lưới và dòng "Chọn: …" vào đây sau khi chạy trên Kaggle.